# Exploring the phase structure function rabbit hole

In [ ]:
from n23_fluctuations_ragland import tools
from matplotlib import pyplot as plt
import scipy
import numpy as np
from tqdm import tqdm

# matplotlib retina setting
%config InlineBackend.figure_format = 'retina'

%load_ext autoreload
%autoreload 2

In [ ]:
ray_ids = ['-7', '+7', '+6', '-6']

In [ ]:
dss = tools.open_receptions()

In [ ]:
wavefront_values = tools.get_wavefront_values(dss, remove_array_tilt=True)

## Phase estimation

In [ ]:
from n23_fluctuations_ragland import estimation

ray_ids = tools.DUCTED_RAY_IDS

# all 28 elements; the phase pattern common to the transmissions (incl. the electronic offsets on elements
# 4, 5, 20) is pooled over every lower-array arrival, as in the estimation experiments' base estimator
est = estimation.Estimator(bad_elements=(), common_phase="all_arrivals")


def unwrapped_phases(wavefronts, est, ray_ids, drop_tx=None):
    """
    Unwrapped phase records (M, n) [rad] of each ray with the common phase removed, following
    `estimation.estimate_spectra`: the common phase is removed as phasors before unwrapping, then as the mean
    of the detrended, unwrapped records. Also returns the number of records in that mean.
    """
    rows = estimation.common_pattern_rows(wavefronts, est, ray_ids, est.common_phase, drop_tx)
    prealigned = estimation.circular_common_phase(wavefronts, est, ray_ids, rows)

    phases, txs = {}, {}
    for ray_id in dict.fromkeys([*ray_ids, *rows]):
        X = wavefronts[ray_id] * np.exp(-1j * prealigned[ray_id])
        phase, _, txs[ray_id] = estimation.wavefront_records(X, est)
        phases[ray_id] = estimation.detrend(phase, est.detrend)

    patterns, n_mean = estimation.common_patterns(phases, txs, rows, est.common_phase, ray_ids)

    out = {}
    for ray_id in ray_ids:
        x = phases[ray_id][txs[ray_id] != drop_tx]
        if patterns[ray_id] is not None:
            x = x - patterns[ray_id]
        out[ray_id] = x
    return out, n_mean


def phase_structure_function(phase, n_mean=np.inf):
    """
    D(k dz) = <(phi(z + k dz) - phi(z))^2> [rad^2], averaged over the transmissions and every element pair at
    lag k (assumes homogeneity along the array). NaN (bad) elements are skipped. Removing a mean of n records
    also removes 1/n of the variance of each one, so D is rescaled by n / (n - 1).
    """
    n = phase.shape[1]
    D = np.zeros(n)
    for k in range(1, n):
        D[k] = np.nanmean((phase[:, k:] - phase[:, :-k]) ** 2)
    if np.isfinite(n_mean):
        D *= n_mean / (n_mean - 1)
    return D


phases, n_mean = unwrapped_phases(wavefront_values, est, ray_ids)
dz_phi = 9 * np.arange(phases[ray_ids[0]].shape[1])  # [m]
D_phi = {r: phase_structure_function(phases[r], n_mean[r]) for r in ray_ids}

# leave-one-transmission-out jackknife (common pattern re-estimated without the dropped transmission)
M = 23
D_phi_jacknife = {r: [] for r in ray_ids}
for k in range(M):
    phases_k, n_mean_k = unwrapped_phases(wavefront_values, est, ray_ids, drop_tx=k)
    for r in ray_ids:
        D_phi_jacknife[r].append(phase_structure_function(phases_k[r], n_mean_k[r]))

D_phi_vars = {}
for r in ray_ids:
    D_phi_jacknife[r] = np.array(D_phi_jacknife[r])
    D_phi_vars[r] = (M - 1) / M * np.sum(
        (D_phi_jacknife[r] - np.mean(D_phi_jacknife[r], axis=0)) ** 2, axis=0
    )

## Mutual coherence estimation

In [ ]:
MC = {}
MC_var = {}

for k, ray_id in enumerate(wavefront_values.keys()):
    dz, MC[ray_id], MC_var[ray_id],_,_ = tools.estimate_mutual_coherence(
        wavefront_values[ray_id], 9, omit_zero_lag=False, norm=False, dt = 1, t0=8/60
    )


In [ ]:
def exp_coherence(dz, z0, A):
    return A * np.exp(-0.5 * (dz / z0) ** 2)

def fit_coherence(wavefont_values, ray_id, jacknife_index=None):
    """
    given estimated coherence, fit an exponential decay to the coherence
    
    Parameters
    ----------
    wavefont_values : dict
        dictionary of wavefront values for each ray_id
    ray_id : str
        ray_id to fit coherence for
    jacknife_index : int, optional
        index of jacknife sample to omit in time dimension, by default None
    """

    def residuals(params, dz, gamma):
        z0, A = params
        return exp_coherence(dz, z0, A) - gamma

    wf_vals = wavefront_values[ray_id]
    if jacknife_index is not None:
        wf_vals = np.delete(wf_vals, jacknife_index, axis=0)
    
    dz, MC, MC_var,_,_ = tools.estimate_mutual_coherence(
        wf_vals, 9, omit_zero_lag=False, norm=False, dt = 1, t0=8/60
    )

    result = scipy.optimize.least_squares(
        residuals,
        x0=[100, 4e11],
        args=(dz[1:], MC[1:]),
        bounds=([0, 0], [np.inf, np.inf]),
        x_scale="jac",  # z0 ~ 1e2 and A ~ 1e11 differ by many orders of magnitude
    )

    z0, A0 = result.x

    return z0, A0, MC, dz

In [ ]:
A0s_jacknife = {}
z0s_jacknife = {}
MCs_jacknife = {}
Ds_jacknife = {}
MCs = {}
Ds = {}
MC_vars = {}
Ds_vars = {}

A0s_all = {}
z0s_all = {}

for ray_id in wavefront_values.keys():
    A0s_arr = []
    z0s_arr = []
    MCs_arr = []
    Ds_arr = []

    for k in range(23):
        z0, A0, MC, dz = fit_coherence(wavefront_values, ray_id, jacknife_index=k)

        A0s_arr.append(A0)
        z0s_arr.append(z0)
        MCs_arr.append(MC)
        Ds_arr.append(-2 * np.log(MC / A0))

    A0s_arr = np.array(A0s_arr)
    z0s_arr = np.array(z0s_arr)
    MCs_arr = np.array(MCs_arr)
    Ds_arr = np.array(Ds_arr)

    z0_all, A0_all, MC_all, dz = fit_coherence(wavefront_values, ray_id)

    A0s_all[ray_id] = A0_all
    z0s_all[ray_id] = z0_all

    Ds_all = -2 * np.log(MC_all / A0_all)

    A0s_jacknife[ray_id] = A0s_arr
    z0s_jacknife[ray_id] = z0s_arr
    MCs_jacknife[ray_id] = MCs_arr
    Ds_jacknife[ray_id] = Ds_arr
    
    MCs[ray_id] = MC_all
    Ds[ray_id] = Ds_all

    M = 23

    MC_vars[ray_id] = (M-1)/M * np.sum((MCs_arr - np.mean(MCs_jacknife[ray_id], axis=0))**2, axis=0) / (A0_all**2)
    Ds_vars[ray_id] = (M-1)/M * np.sum((Ds_arr - np.mean(Ds_arr, axis=0))**2, axis=0)


## Path integral theory

In [ ]:
latitude = 39.192499999
Omega = 7.292115e-5  # Earth Rotation Speed [rad/s]
f = 2 * Omega * np.sin(np.radians(latitude))  # [rad/s]

zeta0 = 7.3 * np.sqrt(1)
rvs = tools.compute_n23_ray_variables(f=f, zeta0=zeta0)

delta_zr = np.linspace(0, 250, 200)

Dzs = {}

#for ray_id in tqdm(wavefront_values.keys()):
for ray_id in tqdm(ray_ids):
    Dzs[ray_id] = []
    for rv in rvs[ray_id]:
        Dzs[ray_id].append(tools.compute_phase_structure_function(rv, delta_zr)[0])

In [ ]:
latitude = 39.192499999
Omega = 7.292115e-5  # Earth Rotation Speed [rad/s]
f = 2 * Omega * np.sin(np.radians(latitude))  # [rad/s]

zeta0 = 7.3 * np.sqrt(1)
rvs = tools.compute_n23_ray_variables(f=f, zeta0=zeta0)

delta_zr = np.linspace(0, 250, 200)

Dzs_nodilation = {}

#for ray_id in tqdm(wavefront_values.keys()):
for ray_id in tqdm(ray_ids):
    Dzs_nodilation[ray_id] = []
    for rv in rvs[ray_id]:
        Dzs_nodilation[ray_id].append(tools.compute_phase_structure_function_no_dilation(rv, delta_zr)[0])

In [ ]:
plt.plot(delta_zr, Dzs_nodilation['-7'][0], label='no dilation')
plt.plot(delta_zr, Dzs['-7'][0], label='with dilation')
plt.legend()

## Weak Fluctuation Theory

In [ ]:
import warnings

# WFT phase structure function: D(dz) = 2 ∫ F_phi(m) [1 - cos(2π m dz)] dm, with m the vertical wavenumber
# [cycles/m] and F_phi the one-sided Colosi (2016) Eq. 6.52 spectrum per cycle/m (`compute_F_phi_chi`,
# <phi^2> = ∫ F_phi dm). Eq. 6.52 is the local straight-ray result, so the separation is dz at every point on
# the ray (no ξ1 dilation, unlike the path integral), and it keeps the Fresnel filter 1 + cos(m² Rf² / 2π)
# (m in rad/m there). Outside the GM band of the ray (below mode 1, above the mode cut-off) compute_F_phi_chi
# returns NaN, which is zero power here. Computed by `tools.compute_wft_structure_function`.
m_wft = np.logspace(-5, 0, 4000)  # [cycles/m], fine enough to resolve cos(2π m dz) at dz = 250 m

D_wft = {}
D_wft_nofresnel = {}
D_wft_chi = {}
D_wft_dilation = {}
F_phi_wft = {}
for ray_id in tqdm(ray_ids):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # NaN/invalid values outside the GM band
        # compute_F_phi_chi takes wavenumbers in rad/m
        F_phi, _ = tools.compute_F_phi_chi(rvs[ray_id][0], 2 * np.pi * m_wft[None, :], f=f)
        F_phi_nofresnel, _ = tools.compute_F_phi_chi(rvs[ray_id][0], 2 * np.pi * m_wft[None, :], f=f, fresnel=False)

        D_wft[ray_id], D_wft_chi[ray_id] = tools.compute_wft_structure_function(
            rvs[ray_id][0], delta_zr, m_wft, f=f
        )

        D_wft_nofresnel[ray_id], _ = tools.compute_wft_structure_function(
            rvs[ray_id][0], delta_zr, m_wft, f=f, fresnel=False
        )

        D_wft_dilation[ray_id], _ = tools.compute_wft_structure_function(
            rvs[ray_id][0], delta_zr, m_wft, f=f, dilation=True, fresnel=False
        )

    F_phi_wft[ray_id] = np.nan_to_num(F_phi)  # [rad^2 m], one-sided per cycle/m

In [ ]:
ray_id = '-7'

plt.plot(D_wft[ray_id], label='WFT with Fresnel')
plt.plot(D_wft_nofresnel[ray_id], label='WFT no Fresnel', linestyle='--')
plt.plot(D_wft_dilation[ray_id], label='WFT with dilation', linestyle=':')
plt.legend()

In [ ]:
# D from the observed F_phi: the same relation as a sum over the periodogram bins m_n = n / (N dz), with
# dm = 1 / (N dz) [cycles/m], of each transmission's Hann-tapered spectrum (one-sided per cycle/m), using the
# same estimator as `D_phi` (all 28 elements, common phase over all arrivals removed). The DC bin is left out
# by `estimate_spectra`, but it adds nothing to D; the Nyquist bin is counted once, since tapered_spectra
# gives it the full one-sided density.
#   - the periodogram is the transform of the circular autocovariance, so D(dz) = D(N dz - dz): only
#     dz <= N dz / 2 = 126 m is meaningful
#   - it is the structure function of the *tapered* record, so scales longer than the array are
#     down-weighted and D at large dz is biased low compared with D_phi
estimates_phi = estimation.estimate_spectra(wavefront_values, est, ray_ids)

n_elem = phases[ray_ids[0]].shape[1]
dm = 1 / (n_elem * 9)  # [cycles/m]
dz_spec = dz_phi[: n_elem // 2 + 1]  # [m]

D_spec = {}
D_spec_vars = {}
for ray_id in ray_ids:
    e = estimates_phi[ray_id]
    m_obs = e.m  # [cycles/m]
    F_phi_obs = e.phi_S  # (M, m) [rad^2 m], one-sided per cycle/m
    w = np.ones(len(m_obs))
    if n_elem % 2 == 0:
        w[-1] = 0.5  # Nyquist bin
    kernel = 1 - np.cos(2 * np.pi * m_obs[None, :] * dz_spec[:, None])  # (dz, m)
    D_tx = 2 * dm * (F_phi_obs * w) @ kernel.T  # (M, dz), one D per transmission

    # D is linear in the spectrum, so the mean of D_tx is D of the mean spectrum, and its leave-one-out
    # jackknife variance is var / M
    M_tx = D_tx.shape[0]
    D_spec[ray_id] = np.mean(D_tx, axis=0)
    D_spec_vars[ray_id] = np.var(D_tx, axis=0, ddof=1) / M_tx

## Results

In [ ]:
fig = plt.figure(figsize=(8,6))
for k, ray_id in enumerate(ray_ids):
    
    plt.subplot(2, 2, k + 1)
    
    plt.errorbar(dz, Ds[ray_id], yerr=np.sqrt(Ds_vars[ray_id]), markersize=2.5, fmt='o', label="$-2 \\  \\mathrm{ln} \\langle p_1 p_2^{\\ast} \\rangle$")
    plt.errorbar(dz_phi, D_phi[ray_id], yerr=np.sqrt(D_phi_vars[ray_id]), markersize=2.5, fmt='o', label="$\\langle (\\phi_1 - \\phi_2)^2 \\rangle$")
    
    plt.plot(delta_zr, Dzs[ray_id][0], label="Path Integral")
    plt.plot(delta_zr, Dzs_nodilation[ray_id][0], label="Path Integral (no dilation)", linestyle='--')

    #plt.plot(delta_zr, D_wft[ray_id], label="WFT")
    #plt.plot(delta_zr, D_wft_nofresnel[ray_id], label="WFT no Fresnel", linestyle='--')
    plt.plot(D_wft_dilation[ray_id], label='WFT with dilation', linestyle=':') 
    
    plt.errorbar(dz_spec, D_spec[ray_id], yerr=np.sqrt(D_spec_vars[ray_id]), markersize=2.5, fmt='s', label="$2\\sum F_\\phi(1-\\cos)$ (Hann)")

    plt.xlabel("$\\Delta z$ [m]")
    plt.ylabel("D $[ \\ rad^2 \\ ]$")

    plt.yscale('log')
    plt.xscale('log')

    plt.xlim([1e0, 3e2])
    plt.title(ray_id, loc='left')
# one legend for all panes, centred below them
fig = plt.gcf()
handles, labels = plt.gca().get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncols=3, bbox_to_anchor=(0.5, 0))
fig.tight_layout(rect=(0, 0.08, 1, 1))  # leave room at the bottom for the legend

In [ ]:
fig = plt.figure(figsize=(8,6))
for k, ray_id in enumerate(ray_ids):
    
    plt.subplot(2, 2, k + 1)
    
    plt.errorbar(dz, Ds[ray_id], yerr=np.sqrt(Ds_vars[ray_id]), markersize=2.5, fmt='o', label="$-2 \\  \\mathrm{ln} \\langle p_1 p_2^{\\ast} \\rangle$")
    plt.errorbar(dz_phi, D_phi[ray_id], yerr=np.sqrt(D_phi_vars[ray_id]), markersize=2.5, fmt='o', label="$\\langle (\\phi_1 - \\phi_2)^2 \\rangle$")
    
    plt.plot(delta_zr, Dzs[ray_id][0], label="Path Integral")
    plt.plot(delta_zr, Dzs_nodilation[ray_id][0], label="Path Integral (no dilation)", linestyle='--')
    plt.plot(D_wft_dilation[ray_id], label='WFT with dilation', linestyle=':') 
    
    plt.plot(delta_zr, D_wft[ray_id], label="WFT")
    plt.errorbar(dz_spec, D_spec[ray_id], yerr=np.sqrt(D_spec_vars[ray_id]), markersize=2.5, fmt='s', label="$2\\sum F_\\phi(1-\\cos)$ (Hann)")

    plt.xlabel("$\\Delta z$ [m]")
    plt.ylabel("D $[ \\ rad^2 \\ ]$")

    #plt.yscale('log')
    #plt.xscale('log')

    plt.xlim([0, 3e2])
    plt.title(ray_id, loc='left')
# one legend for all panes, centred below them
fig = plt.gcf()
handles, labels = plt.gca().get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncols=3, bbox_to_anchor=(0.5, 0))
fig.tight_layout(rect=(0, 0.08, 1, 1))  # leave room at the bottom for the legend